# Label sentences with a model, then chart them

In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
from anthropic import Anthropic

MODEL = "claude-haiku-5-5"

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY before you run this notebook.")

client = Anthropic()

In [ ]:
sentences = [
    "How do I reset my password?",
    "The meeting starts at nine.",
    "Please send me the report.",
    "Where is the nearest station?",
    "The server restarted overnight.",
    "Could you share the slides?",
    "Our team won the prize.",
    "Close the door, please.",
]

numbered = "\n".join(f"{i}. {s}" for i, s in enumerate(sentences, 1))
prompt = (
    "Label each sentence as exactly one of: question, statement, request.\n"
    "Answer with one label per line, in order, and nothing else.\n\n" + numbered
)

response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    thinking={"type": "disabled"},
    messages=[{"role": "user", "content": prompt}],
)
text = "".join(b.text for b in response.content if b.type == "text")
labels = [line.strip().lower() for line in text.splitlines() if line.strip()]

for s, label in zip(sentences, labels):
    print(f"{label:10} {s}")
print("tokens used:", response.usage.input_tokens, "in,", response.usage.output_tokens, "out")

In [ ]:
counts = Counter(labels)
order = ["question", "statement", "request"]
values = [counts.get(name, 0) for name in order]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(order, values, color="#3b6ea5")
ax.set_ylabel("Sentences")
ax.set_title("Sentence types labelled by the model")
ax.set_yticks(range(0, max(values) + 2))
fig.tight_layout()
fig.savefig("chart.png", dpi=120)
plt.show()
print(dict(counts))